# King County housing: refactored notebook

Thin refactored copy of `King-County.ipynb` (ADR-001). Cleaning and features come from `kc_housing` (Phase 3); the modeling cells below are copied verbatim from the reference.

In [ ]:
# Data manipulation, visualization, and machine learning libraries.
import pandas as pd
import plotly.express as px
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from kc_housing.pipeline import run

In [ ]:
# run() rebuilds data/processed/features.parquet
# (raw CSV -> clean -> feature pipeline) and returns its path.
kc_data = pd.read_parquet(run())

*EDA (reference cells 46–76) is left out of this copy for scope. Cells 46–54 need only the clean table; cells 56–76 mix plots of `sqft_price` (a leakage column, computed only for EDA) with the feature code now in `kc_housing` and plots of those features. See King-County.ipynb for the EDA.*

In [ ]:
# Remove columns that would cause data leakage or add no predictive value.
drop_lst = [
    "price",
    "sqft_price",
    "date",
    "delta_lat",
    "delta_long",
]

In [ ]:
# Keep all remaining variables as candidate features.
all_features = [x for x in kc_data.columns if x not in drop_lst]

In [ ]:
# `X` contains the descriptive variables defined above.
X = kc_data[all_features]

In [ ]:
# Define `y` as the target variable.
y = kc_data.price

In [ ]:
# Split the data into training and test sets. We reserve 30% for evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Inspect the size of each split.
print("X_train (features for the model to learn from): ", X_train.shape)
print("y_train (labels for the model to learn from): ", y_train.shape)
print("X_test (features to test the model's accuracy against): ", X_test.shape)
print("y_test (labels to test the model's accuracy with): ", y_test.shape)

In [ ]:
# The training data is now shuffled rather than sorted by index.
X_train.head()

After the split, we can start modeling.
We continue with scikit-learn, where the overall workflow is usually the same:

- import the algorithm
- instantiate the model with its hyperparameters
- choose the feature columns
- fit the model with `.fit(X_train, y_train)`
- evaluate it on the test data

This repeated pattern makes it easy to compare several models in a structured way.



In [ ]:
# Instantiate the model.
model_lin_reg = LinearRegression()

Next, we decide which variables to pass to the model.
We begin with a very simple baseline that uses only one predictor: the house `grade`.



In [ ]:
# Select the variables passed to the model.
variables = ["grade"]

In [ ]:
# Train the model.
model_lin_reg.fit(X_train[variables], y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test[variables], y_test))
        * (X_test.shape[0] - 1)
        / (X_test.shape[0] - len(variables) - 1),
        2,
    ),
)

**R²** measures the proportion of variance in the target variable that is explained by the model.
**Adjusted R²** is a related metric that also accounts for the number of predictors, which makes it more useful when comparing models of different complexity.

Interpretation:
- `1.0` means the model explains all observed variance
- `0.0` means the model explains none of it

In our case, using only `grade` explains about **43%** of the variance in price per square foot on the test set.
That is a useful start, but it suggests that additional variables may improve the model.

Let us inspect the available features again.



In [ ]:
# Show the names of the variables in the dataset.
X_train.columns

We can use more features to predict price per square foot.
The age of the house may also matter, so next we try a linear regression that includes both `grade` and age-related information.



In [ ]:
# Instantiate the model.
model_lin_reg = LinearRegression()

In [ ]:
# Select the variables passed to the model.
variables = ["grade", "last_known_change"]

In [ ]:
# Train the model.
model_lin_reg.fit(X_train[variables], y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test[variables], y_test))
        * (X_test.shape[0] - 1)
        / (X_test.shape[0] - len(variables) - 1),
        2,
    ),
)

With this additional variable, the model explains about **48%** of the variance in price per square foot.



So far, we have only modeled linear relationships between the predictors and price.
However, some effects may be curved rather than straight.

One simple way to capture that is to add squared terms. Instead of fitting only:

$price = b \cdot grade + c$

we can also fit:

$price = a \cdot grade^{2} + b \cdot grade + c$

This is another form of feature engineering. We will apply it to the dataset and see whether the richer representation improves performance.



In [ ]:
# Create only second-order polynomial features (^2).
poly = PolynomialFeatures(2)

In [ ]:
# Create copies of the training and test data.
X_train_poly = X_train.copy()
X_test_poly = X_test.copy()

# Drop the `id` column.
X_train_poly = X_train_poly.drop(columns=["id"])
X_test_poly = X_test_poly.drop(columns=["id"])

In [ ]:
# Create transformed variables with `poly`.
X_train_sq = poly.fit_transform(X_train_poly)

# Apply the same transformation to the test data.
X_test_sq = poly.transform(X_test_poly)

In [ ]:
# Instantiate the model.
model_lin_reg = LinearRegression()

In [ ]:
# Train the model with the squared features.
model_lin_reg.fit(X_train_sq, y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test_sq, y_test))
        * (X_test_sq.shape[0] - 1)
        / (X_test_sq.shape[0] - X_test_sq.shape[1] - 1),
        2,
    ),
)

Adding squared features improved the model a little further.



Adjusted R² gives us one useful summary of model quality, but it is also important to inspect the errors directly.
A **residual analysis** can reveal systematic problems that a single score might hide.  

For easier interpretation, we calculate the percentage difference between the predicted and true prices.



In [ ]:
# Error analysis
# To inspect the model errors more closely, create a new DataFrame with the
# columns `price` (the true value), `lat`, and `long`.
y_predictions = model_lin_reg.predict(X_test_sq)
df_error = pd.DataFrame(y_test)
df_error["latitude"] = X_test["lat"]
df_error["longitude"] = X_test["long"]
df_error["id"] = X_test["id"]
df_error.head(2)

In [ ]:
# Reset the index before adding the predicted price as a new column.
df_error.reset_index(inplace=True, drop=True)
df_error.head(2)

In [ ]:
# Add the predicted price and calculate the difference.
df_error["price_prediction"] = y_predictions.round(2)
df_error["price_difference"] = (df_error["price_prediction"] - df_error["price"]).round(
    2
)
df_error["price_difference_percent"] = (
    (df_error["price_difference"] / df_error["price"]) * 100
).round(2)
df_error.head(2)

In [ ]:
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

We can already see a few strong outliers in the residual plot, so it is worth inspecting the largest one more closely.



In [ ]:
df_error[
    df_error["price_difference_percent"] == df_error["price_difference_percent"].max()
]

In [ ]:
X_test[X_test["id"] == 9272202260]

Let us inspect these houses in more detail.
King County provides a helpful property lookup tool on [this page](https://localscape.property/#kingcountyassessor/My-Property), where you can search by parcel ID.

To inspect the outlier:
- change the selector in the top-left corner from `Address` to `Parcel ID`
- enter the `id` of the outlier

Under **Basic Property Characteristics** and in the **Aerial Images** view, we can see that there is no longer a house on the property. That explains why our model estimated a much higher sale value than the actual transaction suggests.



## Regularization and Hyperparameter Tuning

After adding squared features, the model has many more inputs.
Some of them may contribute useful signal, while others mainly capture noise.
If a model learns noise instead of general patterns, it becomes **overfitted**.

**Regularization** helps control this problem by shrinking coefficients and, in some cases, pushing some of them all the way to zero.
That allows the model to keep the more informative variables while reducing reliance on weaker ones.

Here we use [ElasticNet](https://scikit-learn.org/stable/modules/linear_model.html#elastic-net), which combines L1 and L2 regularization.

Polynomial features have very different scales. A [Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline) applies polynomial expansion and [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) inside each training fold to prevent validation-data leakage and help convergence. Supply the original feature columns without the identifier, not the already expanded matrix.

The compact grid keeps both penalties active; pure L2 regularization is better handled by Ridge. Five-fold validation compares 45 fits using training data only. The held-out test set is reserved for final evaluation.
A [TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html) also standardizes training prices within each fold, making the regularization grid meaningful on a unit-scale target. Predictions are automatically converted back to dollars.

To find a good amount of regularization, we test multiple hyperparameter values and compare the results with [GridSearch](https://scikit-learn.org/stable/modules/grid_search.html#grid-search) and [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation).



In [ ]:
# Specify the hyperparameters to tune.
# `alpha` controls the amount of regularization.
param_grid = {
    "model__regressor__alpha": [0.01, 0.1, 1],
    "model__regressor__l1_ratio": [0.2, 0.5, 0.8],
}

# Keep preprocessing with the estimator for validation and inference.
elastic_pipeline = Pipeline(
    [
        ("polynomial", PolynomialFeatures(2, include_bias=False)),
        ("scaler", StandardScaler()),
        (
            "model",
            TransformedTargetRegressor(
                regressor=ElasticNet(max_iter=5000, tol=1e-4, precompute=True),
                transformer=StandardScaler(),
            ),
        ),
    ]
)

# Pass the model to a parameter grid with 5-fold cross-validation.
elastic_cv = GridSearchCV(
    elastic_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    verbose=2,
    n_jobs=1,
    error_score="raise",
)

# Train and optimize the model with grid search.
elastic_cv.fit(X_train_poly, y_train)

In [ ]:
# Display the best parameters found by the grid search.
elastic_cv.best_params_

In [ ]:
# GridSearchCV already refits the winning pipeline on all training rows.
elastic = elastic_cv.best_estimator_

In [ ]:
# Evaluate how well the model performs on the test data.
print("Test R^2:", round(elastic.score(X_test_poly, y_test), 3))

Compare held-out R^2 with the earlier models; regularization does not guarantee improvement. Counting nonzero coefficients is not an exact degrees-of-freedom correction for ElasticNet, so we report ordinary R^2.  

As discussed above, regularization can set some regression coefficients to zero, effectively removing less useful variables from the model.  

The following code displays part of the learned coefficient vector. Check whether any coefficients became zero rather than assuming they did.



In [ ]:
# Coefficients are in standardized feature and target units, not dollars.
elastic.named_steps["model"].regressor_.coef_[0:5]

In [ ]:
# Error analysis
# To inspect the model errors more closely, create a new DataFrame with the
# columns `price` (the true value), `lat`, and `long`.
y_predictions = elastic.predict(X_test_poly)
df_error = pd.DataFrame(y_test)
df_error["latitude"] = X_test["lat"]
df_error["longitude"] = X_test["long"]
df_error["id"] = X_test["id"]
df_error.head(2)

In [ ]:
# Reset the index before adding the predicted price as a new column.
df_error.reset_index(inplace=True, drop=True)
df_error.head(2)

In [ ]:
# Add the predicted price and calculate the difference.
df_error["price_prediction"] = y_predictions.round(2)
df_error["price_difference"] = (df_error["price_prediction"] - df_error["price"]).round(
    2
)
df_error["price_difference_percent"] = (
    (df_error["price_difference"] / df_error["price"]) * 100
).round(2)
df_error.head(2)

In [ ]:
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

In [ ]:
df_error[
    df_error["price_difference_percent"] == df_error["price_difference_percent"].max()
]

In [ ]:
X_test[X_test["id"] == 5111400086]

Let us take a closer look at this house on [this page](https://localscape.property/#kingcountyassessor/My-Property).

This house sold in 2018 for $295,000. King County appraisal data also reports historical values, and under **Historical Value** we can see that the **Total Assessed Value** in 2014 was $212,000.
That means the sale price was still unusually low relative to the broader context, which helps explain why it stands out so strongly in the residual analysis.



## Save the Model

We have now trained a model that can predict house prices in King County.
The final step is to save the trained pipeline with [skops](https://skops.readthedocs.io/en/stable/modules/classes.html#module-skops.io). Polynomial expansion and scaling are included: inference expects the original training feature columns in the same order, without the identifier.

If you want to explore the optional application stretch goal, see [bonus_solution/](./bonus_solution/). It contains a small working FastAPI + Postgres + Docker example that is meant as a reference, not a required solution path.



In [ ]:
import os

import skops.io as sio

# Create the directory if it does not already exist.
os.makedirs("model", exist_ok=True)

with open("model/model.bin", "wb") as f_out:
    sio.dump(elastic, f_out)

### Conclusion

Despite the presence of outliers, we built a model that explains roughly **83%** of the variance in the target.  

The notebook also shows an important modeling lesson: **feature engineering, nonlinear terms, and regularization can all make a meaningful difference in predictive performance**.

